In [2]:
import numpy as np
from scipy import optimize

### generating data

In [3]:
# function for multivariate hawkes process
# code taken from https://arxiv.org/pdf/2502.18979 Algorithm 1

# mu - d dimensional vector
# alpha - dxd dimensional matrix
# beta - scalar > 0
# time - scalar > 0
def simulation_by_cluster_representation(mu, alpha, beta, time):
    d = len(mu) # dimension

    # family tree is not part of the paper's algorithm
    family_trees = [{} for _ in range(d)] # timestamp of event: (parent variable, timestamp of parent). parents of immigrants are -1

    # initialization
    T = [[] for _ in range(d)] # list of all immigrants and descendants
    A = [[] for _ in range(d)] # temporary list of ancestors

    # immigrant simulation
    for j in range(d):
        k = np.random.poisson(lam=mu[j]*time) # number of immigrants of type j

        small_t = [[] for _ in range(d)] # t != T (from the paper's algorithm)
        for _ in range(k):
            small_t[j].append(np.random.uniform(low=0, high=time)) # small t is different from big T

        A[j] = small_t[j]
        T[j] = list(set(T[j] + A[j]))

        for immigrant in T[j]: # assigns the parent of each immigrant to -1
            family_trees[j][immigrant] = (j, -1)

    # helper function for while loop condition (self explanatory)
    def there_exists_at_least_one_j_st_Aj_neq_empty(A):
        for j in range(len(A)):
            if A[j] != []:
                return True
        return False

    # offspring simulation
    while there_exists_at_least_one_j_st_Aj_neq_empty(A):
        O = [[] for _ in range(d)] # offspring initialization
        for j in range(d):
            if A[j] != []:
                for l in range(len(A[j])):
                    for j_prime in range(d):
                        if alpha[j_prime][j] > 0:
                            k = np.random.poisson(lam=alpha[j_prime][j])
                            
                            # finds elapsed time for descendants
                            small_t = [[] for _ in range(d)]
                            for i in range(k):
                                small_t[j_prime].append(np.random.exponential(beta))

                            # adds elapsed time of descendant to the timestamp of its parent (A[j][l] is the parent)
                            a_plus_t = []
                            for i in range(k):
                                a_plus_t.append(A[j][l] + small_t[j_prime][i])
                            O[j_prime] = list(set(O[j_prime] + a_plus_t))

                            # assigns the parent of each descendant
                            for descendant in a_plus_t:
                                family_trees[j_prime][descendant] = (j, A[j][l])

                        T[j_prime] = list(set(T[j_prime] + O[j_prime])) # offsprings are added to T
        
        A = O # offspring become ancestors

    # remove events beyond T and sort
    for j in range(d):
        i = 0
        while i < len(T[j]): # removes events that occur after time limit
            if T[j][i] > time:
                T[j].pop(i)
                i -= 1
            i += 1
        T[j] = sorted(T[j])

        # remove events from family_tree if they're beyond T
        for key in list(family_trees[j].keys()):
            if key > time:
                family_trees[j].pop(key)
    
    return T, family_trees


In [4]:
# INCORRECT log likelihood

"""
# z[i][j] is 1 if node i is a descendant of node j
#     z[i][0] is 1 if node i is an immigrant
def construct_z(timestamps, family_tree):
    z = [[0 for _ in range(len(timestamps) + 1)] for _ in timestamps]
    for time in family_tree.keys():
        index_of_descendant = timestamps.index(time)
        if family_tree[time] != -1: # if is a descendant
            index_of_parent = timestamps.index(family_tree[time])
            z[index_of_descendant][index_of_parent + 1] = 1
        else: # if not a descendant
            z[index_of_descendant][0] = 1
    return z

# *args contains all_nodes, hyperedges, p[0], family_tree, time, beta
def complete_data_log_likelihood(mu, alpha, beta, time, timestamps, family_tree):
    T = time
    t = timestamps
    z = construct_z(t, family_tree)
    
    total = -mu * T
    n_T = len(t)
    for i in range(n_T):
        total -= -(alpha / beta) * (np.exp(beta * t[i] - beta * T))
        total -= (alpha / beta)
    for i in range(n_T):
        total += (np.log(mu)) * (z[i][0])
    for i in range(n_T):
        for j in range(i):
            total += (z[i][j]) * (np.log(alpha) - beta * (t[i] - t[j]))

    return total
"""

'\n# z[i][j] is 1 if node i is a descendant of node j\n#     z[i][0] is 1 if node i is an immigrant\ndef construct_z(timestamps, family_tree):\n    z = [[0 for _ in range(len(timestamps) + 1)] for _ in timestamps]\n    for time in family_tree.keys():\n        index_of_descendant = timestamps.index(time)\n        if family_tree[time] != -1: # if is a descendant\n            index_of_parent = timestamps.index(family_tree[time])\n            z[index_of_descendant][index_of_parent + 1] = 1\n        else: # if not a descendant\n            z[index_of_descendant][0] = 1\n    return z\n\n# *args contains all_nodes, hyperedges, p[0], family_tree, time, beta\ndef complete_data_log_likelihood(mu, alpha, beta, time, timestamps, family_tree):\n    T = time\n    t = timestamps\n    z = construct_z(t, family_tree)\n    \n    total = -mu * T\n    n_T = len(t)\n    for i in range(n_T):\n        total -= -(alpha / beta) * (np.exp(beta * t[i] - beta * T))\n        total -= (alpha / beta)\n    for i in r

In [166]:
# generate data

# model parameters
mu = [0.1, 0.1] # background intensity
alpha = [[0.3, 0.45],
         [0.35, 0.4]] # mutual excitation matrix; alpha_12 means that 1 is excited by 2
beta = 1 # decay; assume beta is the same for all variables
time = 500 # time

np.random.seed(0)
timestamps, family_trees = simulation_by_cluster_representation(mu, alpha, beta, time)

num_params = len(alpha)

for i in timestamps:
    print(len(i), i)

# for tree in family_trees:
#     print()
#     for key in sorted(tree.keys()):
#         print(key, ':', tree[key])

194 [9.394900218177572, 10.10919872016286, 30.112735814634917, 35.51802909894347, 43.56464985077035, 50.098052040814075, 52.44716258957319, 59.13721293446661, 59.66418455574389, 60.43168394231795, 64.46314882742665, 66.79293261029483, 66.97080937110205, 67.07262501411657, 67.8111501967233, 67.84890811870666, 69.3550527883918, 69.37784698905199, 71.6766437045232, 72.35039657009686, 73.17697633018865, 81.60608073090252, 105.19128053692044, 105.48423199167398, 122.48832394141951, 123.1862614924834, 124.96132185667291, 125.2031842340993, 125.51407629564955, 125.70888239897018, 132.27780605231348, 132.4272371439506, 134.17142293256398, 150.83640108798815, 157.71417546209193, 160.4871900037291, 160.88275559678252, 179.753950286893, 181.8553854713113, 191.72075941288884, 193.85025877761197, 207.3309699952618, 211.82739966945235, 212.96003380364013, 218.51597689967073, 218.79360563134625, 219.0800410506786, 219.30075673116016, 220.02928056562595, 220.3161896583122, 220.38104795002118, 220.4942

### optimizing

In [167]:
# function for log likelihood of multivariate hawkes process
# code taken from Laub's textbook "The Elements of Hawkes Processes" section 5.3 

def generate_timestamp_tuple(timestamps):
    timestamp_tuple = []
    for i in range(len(timestamps)):
        for time in timestamps[i]:
            timestamp_tuple.append((time, i))
    return sorted(timestamp_tuple, key=lambda x: x[0])

# timestamps must be stored as a list of tuples (time, index of point process)
def calculate_intensity_matrix(alpha, timestamps, mu, beta, time):
    l = mu
    for t in timestamps:
        if t[0] < time:
            l = np.add(l, alpha[int(t[1].astype(np.int64))] * np.exp(-beta * (time - t[0]))) # alpha[t[1]] is a 1D matrix
    return l

def calculate_compensator_matrix(alpha, timestamps, mu, beta, time):
    l = mu * time
    for t in timestamps:
        if t[0] < time:
            l = np.add(l, (1/beta) * alpha[int(t[1].astype(np.int64))] * (1 - np.exp(-beta * (time - t[0]))))
    return l

# args = timestamps, mu, beta, time
def negative_log_likelihood(alpha, *args):
    timestamps, mu, beta, time, num_params = args
    timestamp_tuple = generate_timestamp_tuple(timestamps)
    alpha = alpha.reshape((num_params,num_params))

    timestamp_tuple = np.array(timestamp_tuple)
    mu = np.array(mu)
    alpha = np.array(alpha)

    l = 0
    for t in timestamp_tuple:
        intensity = calculate_intensity_matrix(alpha, timestamp_tuple, mu, beta, t[0])
        l += np.log(intensity[int(t[1].astype(np.int64))])
    for k in range(len(mu)):
        compensator = calculate_compensator_matrix(alpha, timestamp_tuple, mu, beta, time)
        l -= compensator[k]
    return -l

In [186]:
# optimize multivariate hawkes parameters
# assume known mu and beta

alpha = np.array(alpha).flatten() # lbfgs requires flattening; will be unflattened later
args = (timestamps, mu, beta, time, num_params)

likelihood = negative_log_likelihood(alpha, *args)
print(likelihood)

initial_guess = [0.5 for _ in alpha]
bounds = []
for i in range(len(alpha)):
    bounds.append((0.0001, 1-0.0001))
bounds = np.array(bounds)

print('\noptimize with scipy-calculated gradient:')
opt_auto_grad = optimize.fmin_l_bfgs_b(func=negative_log_likelihood,
                                       args=args,
                                       x0=initial_guess,
                                       bounds=bounds,
                                       approx_grad=True)
for res in opt_auto_grad:
    print(res)

print('\nactual alpha:')
print(alpha)

499.722290897133

optimize with scipy-calculated gradient:
[0.37219904 0.28270227 0.43399587 0.42409971]
493.65068584685423
{'grad': array([-0.000216  , -0.00052296,  0.00025011, -0.00019327]), 'task': 'CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH', 'funcalls': 60, 'nit': 10, 'warnflag': 0}

actual alpha:
[0.3  0.45 0.35 0.4 ]


### latent positions

In [169]:
# Generate latent variables according to Gaussian distribution with specified variance
# z_j ~ N(0,sigma_z^2 I_d)  Z: shape = (p, d)
def generate_latent_Z(p, d, sigma_z):
    Z = np.random.normal(0, sigma_z, size=(p, d))
    return Z

# Generate Theta matrix with entries distributed according to Gaussian distribution with specified
# variance centered at value determined by pairwise distance between latent positions
# theta_jk = N(alpha - |z_j-z_k|^2,sigma_theta^2)   Theta: shape = (p, p)
def generate_theta(Z, alpha, sigma_theta):
    p = Z.shape[0]
    Theta = np.zeros((p, p))
    for j in range(p):
        for k in range(p):
            Theta[j, k] = np.random.normal(alpha - np.sum((Z[j] - Z[k])**2), sigma_theta)
    return Theta

def softplus(theta):
    theta_tilde = np.copy(theta)
    for i in range(len(theta)):
        for k in range(len(theta[i])):
            theta_tilde[i][k] = np.log(1 + np.exp(theta[i][k]))

    return theta_tilde

In [210]:
# np.random.seed(0)

p = 15 # number of latent nodes to generate
d = 4 # dimension of latent space
sigma_z = 1 # variance of latent space generation

alph = -2 # constant for theta generation; this is alpha but i don't want to confuse it with the hawkes alpha
sigma_theta = 1 # variance for theta generation

Z = generate_latent_Z(p, d, sigma_z)
print(Z)

theta = generate_theta(Z, alph, sigma_theta) # alpha must be small, or else theta_tilde values will be > beta
print(theta)

theta_tilde = softplus(theta)
print(theta_tilde)

t, _ = simulation_by_cluster_representation(mu, theta_tilde, beta, time)
print(t)

[[ 8.73311836e-01  1.19973618e+00  4.56153036e-01 -3.57579032e-01]
 [ 4.10822261e-02  5.93465920e-01  1.01855187e-02  2.19829634e+00]
 [-9.90670931e-01 -1.00266859e+00 -9.76895387e-01 -5.89579923e-01]
 [-2.17893152e+00 -6.29650427e-01 -6.53284702e-01  7.85140252e-02]
 [ 4.17800583e-01 -1.24021634e+00  9.00054243e-01  1.80224223e+00]
 [-2.08285103e-01  1.57437124e+00  1.98989494e-01  1.98873192e+00]
 [ 1.11728347e+00 -1.56390463e+00  1.86273707e-02  1.05432497e+00]
 [ 3.05465810e-02 -3.68835308e-02  1.26976475e+00 -7.09854182e-01]
 [ 1.75156133e-02  3.23625765e-01 -3.33790960e-01 -2.01291039e-02]
 [ 7.75023263e-01  4.32837621e-01 -8.08717532e-01 -1.10412399e+00]
 [-7.89102180e-01  1.24845579e-03 -1.59939788e-01 -8.31957493e-01]
 [-5.98150453e-01 -1.52003929e+00  4.17853703e-01 -4.00187254e-02]
 [-1.25978734e+00  2.86205042e-02  1.34262201e+00 -7.39935853e-01]
 [ 1.31513767e+00 -3.23457472e-01  1.97828168e-01  9.77508024e-02]
 [ 1.40152342e+00  1.58433847e-01 -1.14190142e+00 -1.31097037e

In [213]:
def euclidean_norm(z1, z2):
    total = 0
    for i in range(len(z1)):
        total += (z1[i] - z2[i]) ** 2
    return total

def negative_complete_data_log_likelihood(z, *args):
    p, d, theta, alph, sigma_theta, sigma_z, alpha, timestamps, mu, beta, time, num_params = args
    z = z.reshape(p,d)

    args = (timestamps, mu, beta, time, num_params)
    c = negative_log_likelihood(alpha, *args)

    for j in range(p):
        for k in range(j,p):
            numerator = (theta[j][k] - (alph - euclidean_norm(z[j], z[k]))) ** 2
            denominator = 2 * sigma_theta
            c -= (numerator / denominator) * np.log(np.e / np.sqrt(2 * np.pi * sigma_theta))

    for j in range(p):
        for l in range(d):
            numerator = z[j][l] ** 2
            denominator = 2 * sigma_z
            c -= (numerator / denominator) * np.log(np.e / np.sqrt(2 * np.pi * sigma_z))

    return -c

In [218]:
Z = np.array(Z).flatten()
args = (p, d, theta, alph, sigma_theta, sigma_z, alpha, t, mu, beta, time, int(len(alpha)/2))
ll = negative_complete_data_log_likelihood(Z, *args)
print(ll)

initial_guess = [0.5 for _ in Z]

print('\noptimize with scipy-calculated gradient:')
opt_auto_grad = optimize.fmin_l_bfgs_b(func=negative_complete_data_log_likelihood,
                                       args=args,
                                       x0=initial_guess,
                                       approx_grad=True)
for res in opt_auto_grad:
    print(res)

print('\nactual Z:')
print(Z)

-351.6814350705901

optimize with scipy-calculated gradient:
[-6.15507368e-06 -6.15507368e-06 -6.15507368e-06 -6.15507368e-06
 -6.15507368e-06 -6.15507368e-06 -6.15507368e-06 -6.15507368e-06
 -6.15507368e-06 -6.15507368e-06 -6.15507368e-06 -6.15507368e-06
 -6.15507368e-06 -6.15507368e-06 -6.15507368e-06 -6.15507368e-06
 -6.15507368e-06 -6.15507368e-06 -6.15507368e-06 -6.15507368e-06
 -6.15507368e-06 -6.15507368e-06 -6.15507368e-06 -6.15507368e-06
 -6.15507368e-06 -6.15507368e-06 -6.15507368e-06 -6.15507368e-06
 -6.15507368e-06 -6.15507368e-06 -6.15507368e-06 -6.15507368e-06
 -6.15507368e-06 -6.15507368e-06 -6.15507368e-06 -6.15507368e-06
 -6.15507368e-06 -6.15507368e-06 -6.15507368e-06 -6.15507368e-06
 -6.15507368e-06 -6.15507368e-06 -6.15507368e-06 -6.15507368e-06
 -6.15507368e-06 -6.15507368e-06 -6.15507368e-06 -6.15507368e-06
 -6.15507368e-06 -6.15507368e-06 -6.15507368e-06 -6.15507368e-06
 -6.15507368e-06 -6.15507368e-06 -6.15507368e-06 -6.15507368e-06
 -6.15507368e-06 -6.15507368e